# SynBioCrow 2.2.0-rc1 — All Generators Setup
This notebook installs/imports SynBioCrow plus DORAnet, RetroBioCat2, RetroPath2/rp2paths, RetroTide, and (optionally) BioPKS Pipeline.\n\n**Important:** importable does not mean execution-ready. RBC2 needs its data assets; RetroPath2 needs KNIME + RetroRules + sink data; BioPKS remains an external authorized runtime and requires explicit license acknowledgement.\n

In [ ]:
import os, sys, subprocess, json, time, shutil, importlib, textwrap
from pathlib import Path
T0=time.time()
def status(p,msg): print(f'[SynBioCrow RC1] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {msg}',flush=True)
status(1,'environment')
print(sys.version)


## 1. Install SynBioCrow core + DORAnet + thermodynamics\nThis installs the consolidated branch and the two Python extras that are safe to install directly in the active notebook environment.\n

In [ ]:
repo=Path('/content/SynBioCrow')
if repo.exists(): shutil.rmtree(repo)
status(5,'clone SynBioCrow 2.2 consolidation branch')
subprocess.run(['git','clone','--depth','1','--branch','develop/2.2-engine-consolidation','https://github.com/theiman112860/SynBioCrow.git',str(repo)],check=True)
status(10,'install core + DORAnet + thermo')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)+'[doranet,thermo]'],check=True)


## 2. Install RetroBioCat2\nSynBioCrow uses RetroBioCat2 as a separate research runtime. This cell installs the proven upstream revision without letting it replace the whole scientific environment.\n

In [ ]:
status(20,'install RetroBioCat2 runtime dependencies')
subprocess.run([sys.executable,'-m','pip','install','-q','PyYAML','sqlalchemy','tables','gdown','tqdm'],check=True)
RBC2_REV='c5f32561b1ed99d0701a6dbb472dc448d6c082e8'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--no-build-isolation',f'git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}'],check=True)
print('RetroBioCat2 installed. Real searches still require RBC2 scientific data assets.')


## 3. Install RetroPath2 wrapper + rp2paths\nThe Python packages can be installed here. Real RetroPath2 execution additionally requires KNIME, a RetroRules-ready rules file, and a sink file.\n

In [ ]:
status(35,'install RetroPath2 Python components')
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/retropath2-wrapper.git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/rp2paths.git'],check=True)
print('RetroPath2 Python components installed.')
print('Execution-ready still requires KNIME + RetroRules rules.csv + sink.csv.')


## 4. Install RetroTide\nRetroTide is installed separately as a specialized PKS generator.\n

In [ ]:
status(50,'install RetroTide')
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/JBEI/RetroTide.git'],check=True)


## 5. Optional BioPKS Pipeline installation\nBioPKS Pipeline is **not** installed automatically. Review the upstream license first. Set `ACK_BIOPKS_LICENSE = True` only after you have done so. SynBioCrow still executes BioPKS through the external JSON bridge.\n

In [ ]:
ACK_BIOPKS_LICENSE=False  # change to True only after reviewing https://github.com/JBEI/BioPKS-Pipeline/blob/main/license.txt
BIOPKS_ROOT=Path('/content/BioPKS-Pipeline')
if ACK_BIOPKS_LICENSE:
    status(62,'clone/install BioPKS Pipeline after explicit acknowledgement')
    if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
    subprocess.run(['git','clone','--depth','1','https://github.com/JBEI/BioPKS-Pipeline.git',str(BIOPKS_ROOT)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(BIOPKS_ROOT)],check=True)
    os.environ['SYNBIOCROW_BIOPKS_ACK_LICENSE']='1'
else:
    print('BioPKS installation skipped. Review upstream license and set ACK_BIOPKS_LICENSE=True to install.')


## 6. Import matrix\nThis checks Python importability independently from backend execution readiness.

In [ ]:
status(72,'import matrix')
MODULES={
 'synbiocrow':'synbiocrow',
 'DORAnet':'doranet',
 'RetroBioCat2':'rbc2',
 'RetroPath2 wrapper':'retropath2_wrapper',
 'rp2paths':'rp2paths',
 'RetroTide':'retrotide',
 'BioPKS Pipeline':'biopks_pipeline',
 'eQuilibrator':'equilibrator_api',
}
import_status={}
for label,name in MODULES.items():
    try:
        mod=importlib.import_module(name)
        import_status[label]={'importable':True,'module':getattr(mod,'__file__',None)}
    except Exception as exc:
        import_status[label]={'importable':False,'error':f'{type(exc).__name__}: {exc}'}
print(json.dumps(import_status,indent=2,sort_keys=True))


## 7. SynBioCrow backend readiness\nThis is stricter than importability and reports configuration/resource gaps.

In [ ]:
status(82,'SynBioCrow readiness')
from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
print(json.dumps(engine.backend_readiness(),indent=2,sort_keys=True))
from synbiocrow.bootstrap import bootstrap_advice
for row in bootstrap_advice(engine):
    print(row)


## 8. Configure external resources\nSet these only when you have the actual resources. No placeholder path is treated as execution-ready.\n

In [ ]:
RETROPATH_RULES_FILE=None   # e.g. '/content/drive/MyDrive/.../rules.csv'
RETROPATH_SINK_FILE=None    # e.g. '/content/drive/MyDrive/.../sink.csv'
BIOPKS_BRIDGE=None          # e.g. '/content/BioPKS-Pipeline/synbiocrow_bridge.py'
if BIOPKS_BRIDGE:
    os.environ['SYNBIOCROW_BIOPKS_RUNNER']=BIOPKS_BRIDGE
print('RetroPath execution-ready:', bool(RETROPATH_RULES_FILE and RETROPATH_SINK_FILE))
print('BioPKS bridge configured:', bool(BIOPKS_BRIDGE))


## 9. Run M7/M9 core verification\nThis does not access protected benchmark truth.

In [ ]:
status(90,'run release-readiness gate')
subprocess.run([sys.executable,str(repo/'scripts/m7_release_readiness.py')],check=True)
status(96,'run M9 package check')
subprocess.run([sys.executable,str(repo/'scripts/build_m9_release.py'),'--check'],check=True)
status(100,'all-generators environment setup complete')
